# 02 Generate Synthetic Data And Classifier Experiments

This is the authoritative Task 1 runner. It retrains a GAN per data budget, generates the matching synthetic pool, and writes standardized outputs to `runs/task1/`. On shared CUDA hosts, keep `DEVICE_OVERRIDE="auto"` unless you want to pin a specific GPU. If auto blocks on shared-memory headroom, lower `CUDA_MIN_FREE_GIB_OVERRIDE` cautiously before rerunning.


In [1]:
from pathlib import Path
import os
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from config import Config
from workflow import DEFAULT_TASK1_SCENARIOS, DEFAULT_TASK1_SIZES, run_task1_pipeline

ROOT


PosixPath('/home/arderd/gan-lab-task-1')

In [2]:
# Edit these values before running.
cfg = Config().with_runtime_profile_defaults()
DATA_ROOT = ROOT / "data_final"
TASK1_OUT_ROOT = ROOT / "runs" / "task1"
TASK1_SIZES = DEFAULT_TASK1_SIZES
TASK1_SCENARIOS = DEFAULT_TASK1_SCENARIOS
SYNTH_BATCH_SIZE = 64
STRICT_FID = True

# Shared-GPU overrides. Leave DEVICE_OVERRIDE="auto" to use the visible CUDA
# device with the most free memory. Lower CUDA_MIN_FREE_GIB_OVERRIDE only
# if you want a riskier start threshold on a crowded shared GPU.
DEVICE_OVERRIDE = "auto"  # e.g. "cuda:0" or "cuda:1"
CUDA_MIN_FREE_GIB_OVERRIDE = None  # e.g. 1.5 or 1.0
NUM_WORKERS_OVERRIDE = None

overrides = {}
if DEVICE_OVERRIDE != "auto":
    overrides["device"] = DEVICE_OVERRIDE
if CUDA_MIN_FREE_GIB_OVERRIDE is not None:
    overrides["cuda_min_free_gib"] = CUDA_MIN_FREE_GIB_OVERRIDE
if NUM_WORKERS_OVERRIDE is not None:
    overrides["num_workers"] = NUM_WORKERS_OVERRIDE
if overrides:
    cfg = cfg.with_overrides(**overrides)

{
    "cfg": cfg,
    "resolved_device": cfg.resolve_device(),
    "cuda_min_free_gib": cfg.cuda_min_free_gib,
    "loader_options": cfg.loader_options(),
    "data_root": DATA_ROOT,
    "task1_out_root": TASK1_OUT_ROOT,
    "task1_sizes": TASK1_SIZES,
    "task1_scenarios": TASK1_SCENARIOS,
    "synth_batch_size": SYNTH_BATCH_SIZE,
    "strict_fid": STRICT_FID,
}


{'cfg': Config(data_root=PosixPath('data_final'), out_root=PosixPath('runs'), img_size=64, channels=3, seed=42, z_dim=128, gan_batch=80, gan_epochs=200, gan_lr_g=0.0002, gan_lr_d=0.0001, sample_every=25, ckpt_every=25, fid_every=25, fid_n_samples=2048, fid_enabled=True, fid_reference_split='train', clf_batch=64, clf_epochs=50, clf_lr=0.0003, num_workers=4, persistent_workers=False, prefetch_factor=2, classifier_compile=False, runtime_profile='default', device='auto', cuda_auto_select='most_free', cuda_min_free_gib=2.0, pin_memory=None, allow_tf32=True),
 'resolved_device': 'cuda:0',
 'cuda_min_free_gib': 2.0,
 'loader_options': {'num_workers': 4,
  'pin_memory': True,
  'persistent_workers': False,
  'prefetch_factor': 2},
 'data_root': PosixPath('/home/arderd/gan-lab-task-1/data_final'),
 'task1_out_root': PosixPath('/home/arderd/gan-lab-task-1/runs/task1'),
 'task1_sizes': [100, 200, 400, 800, 1300],
 'task1_scenarios': ['real', 'synth', 'both', 'real_aug'],
 'synth_batch_size': 64,


## Outputs

A full run writes:

- `runs/task1/gan/n*/...`
- `runs/task1/synth/n*/...`
- `runs/task1/clf/all_results.json`
- `runs/task1/pipeline_summary.json`


In [3]:
task1_run = run_task1_pipeline(
    cfg,
    sizes=TASK1_SIZES,
    scenarios=TASK1_SCENARIOS,
    data_root=DATA_ROOT,
    out_root=TASK1_OUT_ROOT,
    synth_batch_size=SYNTH_BATCH_SIZE,
    strict_fid=STRICT_FID,
)

{
    "n_results": len(task1_run["all_results"]),
    "n_pipeline_rows": len(task1_run["pipeline_summary"]),
    "out_root": task1_run["out_root"],
}



=== Task 1 budget: n=100 per class ===
Device: cuda:0
Loader options: {'num_workers': 4, 'pin_memory': True, 'persistent_workers': False, 'prefetch_factor': 2}
Train loader ready: 300 images across 3 classes
FID loader ready: 3900 images from 'train' split


RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
pipeline_summary = task1_run["pipeline_summary"]
pipeline_summary[-1] if pipeline_summary else {}
